# 01 数据 / 预处理 / FCM 区制 / 因果 VMD 可视化
可视化：原始 vs 清洗后功率、NWP-LMD 误差、FCM 软隶属、因果 VMD 多尺度模态。
运行前需先 `bash scripts/preprocess.sh` 生成 `data/processed/`。

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.utils.config import load_config
cfg = load_config('../configs/default.yaml')
SID = 'station00'
df = pd.read_csv(f"../{cfg['data']['processed_dir']}/{SID}.csv")
df['date_time'] = pd.to_datetime(df['date_time'])
print(df.shape); df.head()

In [ ]:
# 功率与白天标记
seg = df.iloc[:96*5]
plt.figure(figsize=(12,3))
plt.plot(seg['date_time'], seg['power'], label='power')
plt.fill_between(seg['date_time'], 0, seg['power'].max(), where=seg['is_day']>0, alpha=0.1, label='is_day')
plt.legend(); plt.title(f'{SID} power (5 days)'); plt.show()

In [ ]:
# NWP-LMD 辐照误差（订正动机）
if 'nwp_globalirrad' in df and 'lmd_totalirrad' in df:
    err = df['nwp_globalirrad'] - df['lmd_totalirrad']
    plt.figure(figsize=(10,3)); plt.hist(err[df['is_day']>0], bins=60)
    plt.title('NWP-LMD irradiance error (day)'); plt.show()
    print('mean', err[df['is_day']>0].mean(), 'std', err[df['is_day']>0].std())

In [ ]:
# FCM 软区制隶属
from src.data.pvod_dataset import fit_normalizer_fcm
n=len(df); i_tr=int(n*cfg['data']['split'][0])
norm, fcm = fit_normalizer_fcm(df.iloc[:i_tr].copy(), cfg['fcm'])
u = fcm.soft_membership(norm.transform(df)[cfg['fcm']['feature_cols']].values)
plt.figure(figsize=(12,3))
for k in range(u.shape[1]):
    plt.plot(u[:96*3, k], label=f'regime {k}')
plt.legend(); plt.title('FCM soft membership (3 days)'); plt.show()

In [ ]:
# 因果 VMD 多尺度模态（首段）
from src.data.causal_vmd import causal_vmd_features
p = norm.transform(df)['power'].values[:600]
nu = causal_vmd_features(p, cfg['vmd']['K_modes'], cfg['vmd']['alpha'], window=192, stride=8)
plt.figure(figsize=(12,4))
for m in range(nu.shape[1]):
    plt.plot(nu[:, m], label=f'mode {m}')
plt.legend(); plt.title('causal VMD modes'); plt.show()